# 02 — Exploring OpenAlex search features

**Goal:** try the search options from the OpenAlex docs on the same claim and see which ones find better evidence.

**Sections:** search modes → query syntax → filters → sort → select → group_by → paging.

Docs: [search](https://developers.openalex.org/guides/searching) · [semantic search](https://help.openalex.org/api/semantic-search/) · [filters](https://developers.openalex.org/guides/filtering) · [sort](https://developers.openalex.org/guides/sort) · [costs](https://help.openalex.org/access/example-costs/)

## 1. Setup
Same as notebook 01, plus a small **helper function** `run()` so each experiment is one line instead of repeating the request code.

In [1]:
import os
import requests

api_key = os.environ.get("OPENALEX_API_KEY")
print("Key loaded:", api_key is not None)  # True/False only, never the key

URL = "https://api.openalex.org/works"
HEADERS = {"Authorization": f"Bearer {api_key}"}  # key in a header, not the URL

claim = "Auditory attention decoding can be decoded using EEG"

Key loaded: True


In [2]:
def run(params, n=5):
    """Send one search to OpenAlex and print a short summary of the top n papers."""
    r = requests.get(URL, params=params, headers=HEADERS, timeout=30)
    print("Status:", r.status_code)
    if r.status_code != 200:
        print(r.text[:500])  # OpenAlex explains what went wrong
        return None
    d = r.json()
    print("Matches:", d["meta"].get("count"), "| cost $:", d["meta"].get("cost_usd"))
    for w in d["results"][:n]:
        print(f'  {w.get("publication_year")} | {w.get("cited_by_count")} cites '
              f'| score {w.get("relevance_score")} | {w.get("display_name")}')
    return d  # return the data so it can be explored further

## 2. Three search modes
Only **one** of these can be used per request.

| Mode | How it matches | Notes |
|---|---|---|
| `search` | keywords, word endings ignored ("decoded" ≈ "decoding") | title, abstract, full text |
| `search.exact` | keywords exactly as written | no stemming |
| `search.semantic` | **meaning** (AI embeddings of title + abstract) | max 50 results, 1 request/s, ~10× cost, `relevance_score` = cosine similarity (0–1) |

In [3]:
d_kw = run({"search": claim})                  # keyword search (as in notebook 01)

Status: 200
Matches: 16765 | cost $: 0.001
  2014 | 948 cites | score 1522.515 | Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG
  2013 | 4205 cites | score 963.96893 | MEG and EEG data analysis with MNE-Python
  2018 | 148 cites | score 688.87744 | A Comparison of Regularization Methods in Forward and Backward Models for Auditory Attention Decoding
  2018 | 2270 cites | score 603.02356 | A review of classification algorithms for EEG-based brain–computer interfaces: a 10 year update
  2012 | 306 cites | score 600.61865 | Decoding Attended Information in Short-term Memory: An EEG Study


In [4]:
d_exact = run({"search.exact": claim})         # exact words, no stemming

Status: 200
Matches: 3264 | cost $: 0.001
  2016 | 311 cites | score 513.7324 | Auditory-Inspired Speech Envelope Extraction Methods for Improved EEG-Based Auditory Attention Detection in a Cocktail Party Scenario
  2022 | 156 cites | score 500.93826 | STAnet: A Spatiotemporal Attention Network for Decoding Auditory Spatial Attention From EEG
  2022 | 216 cites | score 490.45874 | Imagined speech can be decoded from low- and cross-frequency intracranial EEG features
  2021 | 171 cites | score 462.69687 | Electroencephalography-Based Auditory Attention Decoding: Toward Neurosteered Hearing Devices
  2012 | 306 cites | score 453.13803 | Decoding Attended Information in Short-term Memory: An EEG Study


In [5]:
d_sem = run({"search.semantic": claim})        # search by meaning

Status: 200
Matches: 50 | cost $: 0.001
  2016 | 311 cites | score 1.161923755369627 | Auditory-Inspired Speech Envelope Extraction Methods for Improved EEG-Based Auditory Attention Detection in a Cocktail Party Scenario
  2022 | 156 cites | score 1.11331292438271 | STAnet: A Spatiotemporal Attention Network for Decoding Auditory Spatial Attention From EEG
  2018 | 117 cites | score 1.0899775998606973 | EEG-based auditory attention detection: boundary conditions for background noise and speaker positions*
  2022 | 45 cites | score 1.011313375208952 | Detecting the locus of auditory attention based on the spectro-spatial-temporal analysis of EEG
  2016 | 32 cites | score 1.0022548702132599 | Auditory attention decoding with EEG recordings using noisy acoustic reference signals


## 3. Query syntax (for `search`)
- `"..."` exact phrase
- `AND` / `OR` / `NOT` in capitals
- `"word1 word2"~5` words within 5 positions of each other
- `word~1` fuzzy match (allows 1 typo)

In [6]:
x = run({"search": '"auditory attention decoding" EEG'})                      # phrase + word

Status: 200
Matches: 676 | cost $: 0.001
  2021 | 171 cites | score 402.93332 | Electroencephalography-Based Auditory Attention Decoding: Toward Neurosteered Hearing Devices
  2018 | 148 cites | score 384.6034 | A Comparison of Regularization Methods in Forward and Backward Models for Auditory Attention Decoding
  2018 | 49 cites | score 290.09235 | EEG and audio dataset for auditory attention decoding
  2020 | 54 cites | score 251.45494 | Cognitive-Driven Binaural Beamforming Using EEG-Based Auditory Attention Decoding
  2021 | 33 cites | score 203.14546 | Neural tracking to go: auditory attention decoding and saliency detection with mobile EEG


In [7]:
x = run({"search": '("attention decoding" OR "attended speaker") AND EEG'})    # boolean

Status: 200
Matches: 1421 | cost $: 0.001
  2021 | 171 cites | score 372.03485 | Electroencephalography-Based Auditory Attention Decoding: Toward Neurosteered Hearing Devices
  2018 | 148 cites | score 343.12634 | A Comparison of Regularization Methods in Forward and Backward Models for Auditory Attention Decoding
  2016 | 156 cites | score 280.66745 | EEG-Informed Attended Speaker Extraction From Recorded Speech Mixtures With Application in Neuro-Steered Hearing Prostheses
  2019 | 145 cites | score 274.07062 | Comparison of Two-Talker Attention Decoding from EEG with Nonlinear Neural Networks and Linear Methods
  2018 | 49 cites | score 258.42468 | EEG and audio dataset for auditory attention decoding


In [19]:
x = run({"search": '("attention decoding") AND EEG NOT "Cognitive"'})      # exclude a topic

Status: 200
Matches: 399 | cost $: 0.001
  2018 | 117 cites | score 178.23552 | EEG-based auditory attention detection: boundary conditions for background noise and speaker positions*
  2017 | 270 cites | score 169.22449 | Noise-robust cortical tracking of attended speech in real-world acoustic scenes
  2016 | 32 cites | score 149.0428 | Auditory attention decoding with EEG recordings using noisy acoustic reference signals
  2019 | 38 cites | score 146.78635 | Impact of Different Acoustic Components on EEG-Based Auditory Attention Decoding in Noisy and Reverberant Conditions
  2025 | 24 cites | score 137.21602 | Real-time control of a hearing instrument with EEG-based attention decoding


In [9]:
x = run({"search": '"attention EEG"~5'})                                       # proximity

Status: 200
Matches: 18943 | cost $: 0.001
  2013 | 414 cites | score 196.21953 | Recognizing the Degree of Human Attention Using EEG Signals from Mobile Sensors
  1998 | 923 cites | score 193.28242 | Induced alpha band power changes in the human EEG and attention
  2019 | 195 cites | score 145.46782 | A Hierarchical Bidirectional GRU Model With Attention for EEG-Based Emotion Classification
  2001 | 883 cites | score 140.27284 | Human anterior and frontal midline theta and lower alpha reflect emotionally positive state and internalized attention: high-resolution EEG investigation of meditation
  2020 | 61 cites | score 132.63725 | High working memory load impairs selective attention: EEG signatures


## 4. Filters
Filters keep only papers matching **exact conditions**.
`,` = AND · `|` = OR · `!` = NOT · `>` `<` = ranges

In [10]:
x = run({"search": claim,
     "filter": "has_abstract:true,type:article"})             # only articles with an abstract

Status: 200
Matches: 9046 | cost $: 0.001
  2014 | 948 cites | score 1522.515 | Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG
  2013 | 4205 cites | score 963.96893 | MEG and EEG data analysis with MNE-Python
  2018 | 148 cites | score 688.87744 | A Comparison of Regularization Methods in Forward and Backward Models for Auditory Attention Decoding
  2018 | 2270 cites | score 603.02356 | A review of classification algorithms for EEG-based brain–computer interfaces: a 10 year update
  2012 | 306 cites | score 600.61865 | Decoding Attended Information in Short-term Memory: An EEG Study


In [11]:
x = run({"search": claim,
     "filter": "from_publication_date:2020-01-01"})           # only recent papers

Status: 200
Matches: 11294 | cost $: 0.001
  2022 | 156 cites | score 599.5711 | STAnet: A Spatiotemporal Attention Network for Decoding Auditory Spatial Attention From EEG
  2020 | 89 cites | score 592.35425 | Fast EEG-Based Decoding Of The Directional Focus Of Auditory Attention Using Common Spatial Patterns
  2021 | 171 cites | score 534.7638 | Electroencephalography-Based Auditory Attention Decoding: Toward Neurosteered Hearing Devices
  2022 | 216 cites | score 488.53537 | Imagined speech can be decoded from low- and cross-frequency intracranial EEG features
  2021 | 163 cites | score 466.37524 | Decoding Covert Speech From EEG-A Comprehensive Review


In [12]:
x = run({"search": claim,
     "filter": "cited_by_count:>100"})                        # only well-cited papers

Status: 200
Matches: 974 | cost $: 0.001
  2014 | 948 cites | score 1522.515 | Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG
  2013 | 4205 cites | score 963.96893 | MEG and EEG data analysis with MNE-Python
  2018 | 148 cites | score 688.87744 | A Comparison of Regularization Methods in Forward and Backward Models for Auditory Attention Decoding
  2018 | 2270 cites | score 603.02356 | A review of classification algorithms for EEG-based brain–computer interfaces: a 10 year update
  2012 | 306 cites | score 600.61865 | Decoding Attended Information in Short-term Memory: An EEG Study


## 5. Sort
Default = `relevance_score` (highest first). `:desc` = descending.

In [13]:
x = run({"search": claim, "sort": "cited_by_count:desc"})         # most cited first

Status: 200
Matches: 16765 | cost $: 0.001
  2021 | 5155 cites | score None | A Survey of Convolutional Neural Networks: Analysis, Applications, and Prospects
  2013 | 4205 cites | score None | MEG and EEG data analysis with MNE-Python
  2012 | 3282 cites | score None | Tracking Whole-Brain Connectivity Dynamics in the Resting State
  2013 | 3207 cites | score None | Dynamic functional connectivity: Promise, issues, and interpretations
  2012 | 2380 cites | score None | The brain basis of emotion: A meta-analytic review


In [14]:
x = run({"search": claim, "sort": "publication_date:desc"})       # newest first

Status: 200
Matches: 16765 | cost $: 0.001
  2026 | 0 cites | score None | Simultaneous Cortical Responses to Multiple Written Words
  2026 | 0 cites | score None | Knowing and feeling: The role of semantic valence in emotional prosody perception for L2 Chinese learners
  2026 | 0 cites | score None | Comparing serial and parallel language input: Shared neural representations with distinct encoding strengths
  2026 | 0 cites | score None | Domain-specific cognitive-motor interference across goal-directed reaching, head-hand coordination, posture and gait: a virtual reality study
  2026 | 0 cites | score None | Auditory attention decoding from Ear-EEG signals: A dataset with dynamic attention switching and rigorous cross-validation


## 6. Select — ask only for the fields I need
Each full paper is large (~32 KB in notebook 01). `select` returns only the named top-level fields.

In [15]:
full = requests.get(URL, params={"search": claim, "per_page": 10},
                    headers=HEADERS, timeout=30)
small = requests.get(URL, params={"search": claim, "per_page": 10,
                                  "select": "id,doi,display_name,publication_year,cited_by_count"},
                     headers=HEADERS, timeout=30)
print("Full response KB:  ", round(len(full.content) / 1024, 1))
print("Select response KB:", round(len(small.content) / 1024, 1))
print(small.json()["results"][0])  # what one paper looks like now

Full response KB:   321.4
Select response KB: 3.0
{'id': 'https://openalex.org/W2158904676', 'doi': 'https://doi.org/10.1093/cercor/bht355', 'display_name': 'Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG', 'publication_year': 2014, 'cited_by_count': 948}


## 7. Group by — count instead of list
`group_by` returns **counts per category** (e.g. per year) instead of papers. Useful to see how a topic grew over time.

In [16]:
g = requests.get(URL, params={"search": claim, "group_by": "publication_year"},
                 headers=HEADERS, timeout=30).json()
for row in sorted(g["group_by"], key=lambda r: str(r.get("key")))[-15:]:  # last 15 years
    print(row.get("key_display_name"), row.get("count"))

2012 271
2013 348
2014 431
2015 439
2016 580
2017 652
2018 831
2019 902
2020 1134
2021 1323
2022 1362
2023 1822
2024 1893
2025 2136
2026 1624


## 8. Paging
`per_page` (max 100) + `page` = which slice of the results.

In [17]:
x = run({"search": claim, "per_page": 5, "page": 2})              # papers 6–10

Status: 200
Matches: 16765 | cost $: 0.001
  2022 | 156 cites | score 599.5711 | STAnet: A Spatiotemporal Attention Network for Decoding Auditory Spatial Attention From EEG
  2020 | 89 cites | score 592.35425 | Fast EEG-Based Decoding Of The Directional Focus Of Auditory Attention Using Common Spatial Patterns
  2019 | 145 cites | score 590.1436 | Comparison of Two-Talker Attention Decoding from EEG with Nonlinear Neural Networks and Linear Methods
  2015 | 385 cites | score 587.36 | EEG Source Imaging Enhances the Decoding of Complex Right-Hand Motor Imagery Tasks
  2016 | 311 cites | score 559.667 | Auditory-Inspired Speech Envelope Extraction Methods for Improved EEG-Based Auditory Attention Detection in a Cocktail Party Scenario
